# Silent sensor failure: a HIL bug, fixed in SIL

Companion notebook for the [Progressive Testing](https://gtcloudrobotics.com/progressive-testing/) page.

Setup: the satellite has two redundant temperature sensors. We average them each step and flag a fault if the average runs hot. The first version of the monitor passed every SIL test we wrote for it — and then quietly let the satellite cook on the bench when one sensor's driver entered a known last-cached-value failure mode.

This notebook walks through:

1. Reproducing the HIL bug in software.
2. Diagnosing it from the trace.
3. Fixing the monitor.
4. Writing the SIL test we *should* have written from the start.

**You are expected to edit cells.** Each section ends with a prompt — try it before scrolling to the next cell.

## 1. The first-pass monitor

Two sensors, averaged, threshold check. Looks fine.

In [2]:
class ThermalMonitor:
    def __init__(self, a, b, max_c=80.0):
        self.a, self.b, self.max_c = a, b, max_c
        self.state = 'NOMINAL'

    def tick(self):
        avg = (self.a.read() + self.b.read()) / 2
        if avg > self.max_c:
            self.state = 'THERMAL_SAFE_MODE'
        else:
            self.state = 'NOMINAL'

## 2. The SIL test that gave us false confidence

Two healthy sensors, average is well below the threshold, monitor stays nominal. Green check, ship it.

In [3]:
class FakeSensor:
    def __init__(self, value): self.value = value
    def read(self): return self.value

def test_monitor_stays_nominal_when_cool():
    a = FakeSensor(42.0)
    b = FakeSensor(43.0)
    monitor = ThermalMonitor(a, b)
    monitor.tick()
    assert monitor.state == 'NOMINAL', monitor.state
    print('SIL passed — monitor stayed nominal under healthy sensors')

test_monitor_stays_nominal_when_cool()

SIL passed — monitor stayed nominal under healthy sensors


## 3. Reproducing the HIL bug

On the bench, sensor A's driver hit a known failure mode: when the device stopped responding, the driver silently returned its last cached reading instead of raising. So in software-land, A is happy. In reality, A is dead and B is watching the satellite heat up alone.

We model that failure with a `FrozenAfter` sensor — it returns real readings until step `freeze_at`, then sticks at its last value forever. Run the cell and watch the trace.

In [4]:
class FrozenAfter:
    """Returns scripted readings; freezes at last value after `freeze_at` calls (last-cached failure mode)."""
    def __init__(self, readings, freeze_at=None):
        self.readings = list(readings)
        self.freeze_at = freeze_at
        self.calls = 0
        self.last = readings[0]
    def read(self):
        if self.freeze_at is not None and self.calls >= self.freeze_at:
            return self.last
        self.last = self.readings[min(self.calls, len(self.readings) - 1)]
        self.calls += 1
        return self.last

# Sensor A freezes after step 0; sensor B reports the satellite genuinely overheating.
a = FrozenAfter([42.1, 42.1, 42.1, 42.1], freeze_at=1)
b = FrozenAfter([43.0, 58.7, 74.4, 91.2])
monitor = ThermalMonitor(a, b)

for t in range(4):
    monitor.tick()
    avg = (a.last + b.last) / 2
    print(f't={t}.0s  A={a.last:>5.1f}  B={b.last:>5.1f}  avg={avg:>5.1f}  state={monitor.state}')

t=0.0s  A= 42.1  B= 43.0  avg= 42.5  state=NOMINAL
t=1.0s  A= 42.1  B= 58.7  avg= 50.4  state=NOMINAL
t=2.0s  A= 42.1  B= 74.4  avg= 58.2  state=NOMINAL
t=3.0s  A= 42.1  B= 91.2  avg= 66.7  state=NOMINAL


Read the trace. The satellite is at 91 °C by t=3 and the monitor reports `NOMINAL` the entire time. The dead sensor is dragging the average down so the threshold check never trips.

**Two redundant sensors made things worse, not better — the redundancy masked the failure.**

The bug isn't the threshold. It isn't the average. It's that the monitor has no way to know one of its inputs is dead.

## 4. Your turn: fix the monitor

A working sensor reports a fresh reading every tick. A frozen sensor reports the same value forever. So the monitor needs each sensor to *tell it* when its reading is fresh, and the monitor should only average sensors that are actually live.

Below is a sensor interface that exposes both `read()` and `is_fresh()`. Edit `ThermalMonitorV2.tick` to:

1. Drop sensors that aren't fresh from the average.
2. If *both* sensors are stale, enter `THERMAL_SAFE_MODE` (we've lost observability — fail safe).
3. Otherwise, threshold-check the average of whichever sensors are live.

Then re-run the failure scenario. The monitor should enter `THERMAL_SAFE_MODE` as soon as B crosses 80 °C — not silently watch the satellite cook.

In [ ]:
class FreshnessAwareSensor:
    """Same scripted readings, but exposes is_fresh() — returns False once frozen."""
    def __init__(self, readings, freeze_at=None):
        self.readings = list(readings); self.freeze_at = freeze_at
        self.calls = 0; self.last = readings[0]; self.fresh = True
    def read(self):
        if self.freeze_at is not None and self.calls >= self.freeze_at:
            self.fresh = False
            return self.last
        self.last = self.readings[min(self.calls, len(self.readings) - 1)]
        self.calls += 1
        self.fresh = True
        return self.last
    def is_fresh(self): return self.fresh

class ThermalMonitorV2:
    def __init__(self, a, b, max_c=80.0):
        self.a, self.b, self.max_c = a, b, max_c
        self.state = 'NOMINAL'

    def tick(self):
        # TODO: read both sensors, drop the stale ones, fail safe if both are stale,
        # otherwise threshold-check the average of the live readings.
        raise NotImplementedError('your turn')

a = FreshnessAwareSensor([42.1, 42.1, 42.1, 42.1], freeze_at=1)
b = FreshnessAwareSensor([43.0, 58.7, 74.4, 91.2])
monitor = ThermalMonitorV2(a, b)

for t in range(4):
    monitor.tick()
    print(f't={t}.0s  A={a.last:>5.1f} (fresh={a.is_fresh()})  B={b.last:>5.1f} (fresh={b.is_fresh()})  state={monitor.state}')

<details>
<summary><b>Reference solution</b> — try the exercise above before clicking</summary>
<pre><code class="language-python">def tick(self):
    fresh_readings = []
    for sensor in (self.a, self.b):
        reading = sensor.read()
        if sensor.is_fresh():
            fresh_readings.append(reading)
    if not fresh_readings:
        self.state = 'THERMAL_SAFE_MODE'   # no observability — fail safe
        return
    avg = sum(fresh_readings) / len(fresh_readings)
    if avg > self.max_c:
        self.state = 'THERMAL_SAFE_MODE'
    else:
        self.state = 'NOMINAL'
</code></pre>
<p>Note that we read every sensor every tick, even ones we end up dropping — <code>read()</code> is what updates <code>is_fresh()</code>. Skipping a sensor because it was stale last tick would mean it can never recover.</p>
</details>

## 5. The SIL test we should have written from the start

Now that we know `last-cached-on-driver-failure` is a real failure mode in this hardware, it costs nothing to keep checking for it on every commit. Run this against your fixed monitor — it should pass.

In [ ]:
def test_safe_mode_when_redundant_sensor_is_silently_frozen():
    a = FreshnessAwareSensor([42.1, 42.1, 42.1, 42.1], freeze_at=1)
    b = FreshnessAwareSensor([43.0, 58.7, 74.4, 91.2])
    monitor = ThermalMonitorV2(a, b)

    states = []
    for _ in range(4):
        monitor.tick()
        states.append(monitor.state)

    # By step 3, B is reporting 91 °C alone — monitor must trip even though A 'looks' fine.
    assert states[-1] == 'THERMAL_SAFE_MODE', states
    print('SIL passed — monitor caught the masked overheat:', states)

test_safe_mode_when_redundant_sensor_is_silently_frozen()

## Takeaway

The original SIL test wasn't wrong — it tested what the author believed could go wrong. The HIL run taught us about a failure mode the author hadn't imagined. Once a failure mode is known, the cheapest place to catch it forever is back in SIL.

That's the whole loop:

- **SIL** catches what you can imagine.
- **HIL and hardware** teach you what you couldn't.
- Every lesson from HIL becomes a new SIL test, so the next regression is caught in milliseconds instead of on a workbench.